# Phase 3 — ViRGo-SAGE: GNN encoder over the virtual graphs

Trains an **unsupervised GraphSAGE** (`encoder.py`) on the **saved Phase-2 virtual graphs**; evaluates node classification + leakage-free link prediction. Only the encoder changes vs Phase 2: Skipgram lookup table → message passing.

- **Reads, never builds:** virtual graphs from `output/notebook2_create_vir_graph/virtual_graphs/<dataset>/k<K>/<sim>/`, LP splits from `splits/link_prediction/virtual_graph_study/`. Missing ⇒ run notebook 2 first.

Design: `docs/phase3_gnn_design.md` · Spine: README Phase 3.


In [ ]:
"""Setup: repo root, config, knobs. This notebook reads Phase-2 artifacts; it builds no graphs."""
import os, sys
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)                     # run everything from the repo root
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

from virgo import config as cfg
from virgo import graph_io
from virgo.eval import nodeclass as eval_nodeclass, linkpred as eval_linkpred, results_io
from virgo.encoders import SageEncoder
from virgo.virtual_graph import VirtualGraph

# THE batch knob - same list as notebook 2's RUN_DATASETS, so both notebooks cover exactly the same graphs.
# Sections 7 and 8 loop over it. Add the five extra datasets that already have scoreboard rows
# ("citeseer_linqs", "enzymes", "lastfm_asia", "minesweeper", "proteins") only AFTER notebook 2 has built their
# hybrid_degree / hybrid_centrality virtual graphs - section 8 skips (never crashes on) anything not yet built.
RUN_DATASETS = ["actor", "amazon_photo", "amazon_ratings", "cora", "pubmed", "questions",
                "roman_empire", "squirrel_filtered", "tolokers"]
DATASET = RUN_DATASETS[0]                           # <-- ONE dataset for the spine sections 1-6 and ablation 8b (must be a single name)
SIM, K = "psi", 10                                  # <-- ONE graph for the spine sections 1-6 (K must be a single int); the K sweep has its own knobs: SWEEP_KS (section 8), SHOW_K (7), XD_K (9), RQ_K (11)
SEEDS = [42, 43, 44]                                # <-- pick seeds manually; keep [42, 43, 44] to match the bridge for fair comparison
POSITIVES = "edge"                                  # <-- ablation A (DECIDED: edge won): "edge" = direct virtual edges | "walk" = A1 walk co-occurrence
AGG = "mean"                                        # <-- ablation B (DECIDED: mean won): "mean" | "weighted" | "sum" | "max"
FEATURES = "all"                                    # <-- ablation D input features (D0 default): "all" | "degree" | "deg_cent" | "psi" | "random" (control) | "const"
P = cfg.GNN_PARAMS                                  # single source of truth (virgo/config.py) - loads GNN settings
FORCE_REBUILD = False                   
# False = reuse saved .emb if present

# Output zones (notebook-first layout): folder names the notebook, task, dataset, K, variant; file names the encoder + seed.
NB2 = Path("output") / "notebook2_create_vir_graph"       # read-only here: virtual graphs + DeepWalk bridge embeddings
NB3 = Path("output") / "notebook3_gnn_encoder"            # this notebook's embeddings
LP_SPLITS = Path("splits") / "link_prediction" / "virtual_graph_study"   # shared per-seed split folders (made by notebook 2)

ENCODER = (("graphsage_walk" if POSITIVES == "walk" else "graphsage_edge") + ("" if AGG == "mean" else f"_{AGG}")
           + ("" if FEATURES == "all" else f"_feat_{FEATURES}"))
# ENCODER is both the .emb filename prefix and the scoreboard name — every ablation writes its own files, nothing overwrites.
print(f"batch={len(RUN_DATASETS)} datasets: {RUN_DATASETS}")
print(f"spine dataset={DATASET}  sim={SIM}  K={K}  seeds={SEEDS}  positives={POSITIVES}  agg={AGG}  features={FEATURES}  ({ENCODER})")
print(P)

batch=9 datasets: ['actor', 'amazon_photo', 'amazon_ratings', 'cora', 'pubmed', 'questions', 'roman_empire', 'squirrel_filtered', 'tolokers']
spine dataset=actor  sim=psi  K=10  seeds=[42, 43, 44]  positives=edge  agg=mean  features=all  (graphsage_edge)
{'hidden': 64, 'dimensions': 64, 'layers': 2, 'agg': 'mean', 'lr': 0.01, 'epochs': 50, 'negatives': 5, 'pairs_per_epoch': 100000, 'max_pairs': 2000000, 'positives': 'edge', 'features': 'all'}


/home/m-adam/miniconda/envs/i2v/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1 · Load the graphs

Original graph from `input/` (node set + features); **saved** virtual graph from Phase 2. Missing file ⇒ run notebook 2.


In [2]:
"""Load original graph + the SAVED Phase-2 virtual graph."""
G = graph_io.load_graph(cfg.dataset(DATASET)["edgelist"])   # shared loader: the same graph Phase 2 built from

vpath = NB2 / "virtual_graphs" / DATASET / f"k{K}" / SIM / "virtual_graph.edgelist"
assert vpath.exists(), f"{vpath} missing -> run notebooks/2-phase_2_virtual_graph.ipynb first (it builds + saves the virtual graphs)"
V = nx.read_weighted_edgelist(vpath, nodetype=int)
V.add_nodes_from(G.nodes)                           # edgelists omit isolated nodes -> restore the full node set
print(f"original: {G.number_of_nodes()} nodes / {G.number_of_edges()} edges")
print(f"virtual({SIM}, K={K}): {V.number_of_nodes()} nodes / {V.number_of_edges()} edges  <- {vpath}")

original: 7600 nodes / 26659 edges
virtual(psi, K=10): 7600 nodes / 47460 edges  <- output/notebook2_create_vir_graph/virtual_graphs/actor/k10/psi/virtual_graph.edgelist


## 2 · Train (GraphSAGE → embeddings)

- `POSITIVES` — ablation A, **decided: `edge`** (positives = virtual edges; `walk` = Phase-2-comparable corpus).
- `AGG` — ablation B, **decided: `mean`** (beat weighted / sum / max).
- `FEATURES` — ablation D, all features for GraphSAGE


In [3]:
"""Train ViRGo-SAGE per seed on the full virtual graph; reuse saved .emb when present."""
losses = None
for seed in SEEDS:
    out = NB3 / "node_classification" / DATASET / f"k{K}" / SIM / f"{ENCODER}_s{seed}.emb"
    if out.exists() and not FORCE_REBUILD:
        print(f"reuse {out}")
        continue
    print(f"train seed {seed} ({POSITIVES} positives, {AGG} agg, {FEATURES} features) ...")
    enc = SageEncoder(G, V, seed, hidden=P["hidden"], dimensions=P["dimensions"], layers=P["layers"], agg=AGG, feats=FEATURES)
    losses = enc.train(P["epochs"], lr=P["lr"], negatives=P["negatives"],
                       pairs_per_epoch=P["pairs_per_epoch"], max_pairs=P["max_pairs"], positives=POSITIVES)
    enc.save(out)
    print(f"saved {out}")

if losses:                                          # spine check: the loss must fall
    plt.figure(figsize=(5, 3))
    plt.plot(losses)
    plt.xlabel("epoch"); plt.ylabel("skipgram-analog loss"); plt.title(f"ViRGo-SAGE  {DATASET}  {SIM}  K={K}  {POSITIVES}  {AGG}  {FEATURES}")
    plt.tight_layout(); plt.show()

reuse output/notebook3_gnn_encoder/node_classification/actor/k10/psi/graphsage_edge_s42.emb
reuse output/notebook3_gnn_encoder/node_classification/actor/k10/psi/graphsage_edge_s43.emb
reuse output/notebook3_gnn_encoder/node_classification/actor/k10/psi/graphsage_edge_s44.emb


## 3 · Verify the embeddings

Every node embedded · all values finite · readable by the same gensim loader the evals use.


In [4]:
"""Spine checks: node count, finiteness, loader compatibility."""
from gensim.models import KeyedVectors
for seed in SEEDS:
    out = NB3 / "node_classification" / DATASET / f"k{K}" / SIM / f"{ENCODER}_s{seed}.emb"
    kv = KeyedVectors.load_word2vec_format(str(out))
    assert len(kv) == G.number_of_nodes(), f"{out.name}: {len(kv)} vectors != {G.number_of_nodes()} nodes"
    assert np.isfinite(kv.vectors).all(), f"{out.name}: non-finite values"
    print(f"PASS {out.name}: {len(kv)} nodes x {kv.vectors.shape[1]}d, finite")

PASS graphsage_edge_s42.emb: 7600 nodes x 64d, finite
PASS graphsage_edge_s43.emb: 7600 nodes x 64d, finite
PASS graphsage_edge_s44.emb: 7600 nodes x 64d, finite


## 4 · Node classification (weighted F1)

One-vs-rest logistic regression on the embeddings, stratified 70/30, per seed — protocol identical to Phase 2 / I2V.


In [5]:
"""Node classification on the SAGE embeddings — protocol identical to Phase 2."""
labels = cfg.dataset(DATASET)["labels"]
nc_rows = []
if labels is None or not Path(labels).exists():
    print(f"{DATASET}: no labels -> node classification skipped")
else:
    for seed in SEEDS:
        emb = NB3 / "node_classification" / DATASET / f"k{K}" / SIM / f"{ENCODER}_s{seed}.emb"
        f1s, n, c = eval_nodeclass.evaluate(str(emb), str(labels), train_frac=cfg.REPRO["nodeclass_train_frac"], seed=seed)
        nc_rows.append({"encoder": ENCODER, "sim": SIM, "K": K, "seed": seed, "task": "nodeclass", "metric": f1s["weighted"]})
        print(f"seed {seed}: weighted F1 = {f1s['weighted']:.4f}  (n={n}, classes={c})")
    vals = [r["metric"] for r in nc_rows]
    print(f"\nNC weighted F1 = {np.mean(vals):.4f} ± {np.std(vals):.4f}")

seed 42: weighted F1 = 0.1709  (n=7600, classes=5)
seed 43: weighted F1 = 0.1744  (n=7600, classes=5)
seed 44: weighted F1 = 0.1619  (n=7600, classes=5)

NC weighted F1 = 0.1691 ± 0.0053


## 5 · Link prediction (AUC, leakage-free)

Same Phase-2 splits. Per seed: 70% `train.edgelist` → deterministically rebuild the train virtual graph → train SAGE on it → score held-out edges. Test edges never touch training.


In [6]:
"""Leakage-free LP: same Phase-2 splits, train-virtual rebuilt from the 70% edges, SAGE trained on it."""
lp_rows = []
for seed in SEEDS:
    split_dir = LP_SPLITS / DATASET / f"seed_{seed}"
    train_edges = split_dir / "train.edgelist"
    assert train_edges.exists(), f"{train_edges} missing -> run the Phase-2 notebook LP section first (shared splits = fair comparison)"
    emb = NB3 / "link_prediction" / DATASET / f"k{K}" / SIM / f"{ENCODER}_s{seed}.emb"
    if not emb.exists() or FORCE_REBUILD:
        print(f"train LP seed {seed} ({POSITIVES} positives, {AGG} agg, {FEATURES} features) ...")
        Gt = graph_io.load_graph(train_edges)
        Vt = VirtualGraph(Gt, seed=cfg.REPRO["seed"]).build(SIM, K)   # from the 70% train edges only; VG seed = project seed, NOT the loop seed
        enc = SageEncoder(Gt, Vt, seed, hidden=P["hidden"], dimensions=P["dimensions"], layers=P["layers"], agg=AGG, feats=FEATURES)
        enc.train(P["epochs"], lr=P["lr"], negatives=P["negatives"],
                  pairs_per_epoch=P["pairs_per_epoch"], max_pairs=P["max_pairs"], positives=POSITIVES)
        enc.save(emb)
    else:
        print(f"reuse {emb}")
    auc = eval_linkpred.evaluate(str(emb), str(split_dir), seed=seed, score=cfg.REPRO["linkpred_score"])
    lp_rows.append({"encoder": ENCODER, "sim": SIM, "K": K, "seed": seed, "task": "linkpred", "metric": auc})
    print(f"seed {seed}: AUC = {auc:.4f}")
vals = [r["metric"] for r in lp_rows]
print(f"\nLP AUC = {np.mean(vals):.4f} ± {np.std(vals):.4f}")

reuse output/notebook3_gnn_encoder/link_prediction/actor/k10/psi/graphsage_edge_s42.emb


seed 42: AUC = 0.6422
reuse output/notebook3_gnn_encoder/link_prediction/actor/k10/psi/graphsage_edge_s43.emb
seed 43: AUC = 0.6508
reuse output/notebook3_gnn_encoder/link_prediction/actor/k10/psi/graphsage_edge_s44.emb
seed 44: AUC = 0.6469

LP AUC = 0.6466 ± 0.0035


## 6 · DeepWalk (Phase 2) vs GraphSAGE (Phase 3)

Both encoders scored **here, the same way**, from their saved `.emb` files — same virtual graph, same splits, same seeds.

`improvement` = GraphSAGE − DeepWalk.

`winner` = higher score.

Missing file ⇒ the cell names the notebook to run. Saved to `results/snapshots/<dataset>_encoder_comparison_K<K><suffix>.csv`; rows also recorded to the scoreboard.


In [7]:
"""One final table: DeepWalk (Phase 2) vs GraphSAGE (Phase 3), both scored here from their saved embeddings."""
labels = cfg.dataset(DATASET)["labels"]
have_labels = labels is not None and Path(labels).exists()
NC, LP = "node classification (weighted F1)", "link prediction (AUC)"

def task_scores(zone, encoder_name, notebook):
    """Score one encoder's saved .emb files with the shared eval protocol -> {task: [score per seed]}."""
    got = {NC: [], LP: []}
    for seed in SEEDS:
        emb = {"node_classification": zone / "node_classification" / DATASET / f"k{K}" / SIM / f"{encoder_name}_s{seed}.emb",
               "link_prediction":     zone / "link_prediction" / DATASET / f"k{K}" / SIM / f"{encoder_name}_s{seed}.emb"}
        if have_labels:
            assert emb["node_classification"].exists(), f"{emb['node_classification']} missing -> run {notebook} with the same DATASET/SIM/K/SEEDS first"
            f1s, _, _ = eval_nodeclass.evaluate(str(emb["node_classification"]), str(labels), train_frac=cfg.REPRO["nodeclass_train_frac"], seed=seed)
            got[NC].append(f1s["weighted"])
        assert emb["link_prediction"].exists(), f"{emb['link_prediction']} missing -> run {notebook} with the same DATASET/SIM/K/SEEDS first"
        got[LP].append(eval_linkpred.evaluate(str(emb["link_prediction"]), str(LP_SPLITS / DATASET / f"seed_{seed}"),
                                              seed=seed, score=cfg.REPRO["linkpred_score"]))
    return got

bridge = task_scores(NB2, "deepwalk", "notebooks/2-phase_2_virtual_graph.ipynb (sections 7-8)")
sage = task_scores(NB3, ENCODER, "sections 2-5 above (same POSITIVES/AGG knobs)")

comp = pd.DataFrame([{"task": t,
                      "deepwalk_phase2": round(float(np.mean(bridge[t])), 4),
                      f"{ENCODER}_phase3": round(float(np.mean(sage[t])), 4),
                      "improvement": round(float(np.mean(sage[t]) - np.mean(bridge[t])), 4),
                      "winner": ENCODER if np.mean(sage[t]) > np.mean(bridge[t]) else "deepwalk"}
                     for t in (NC, LP) if sage[t]])
out = Path("results") / "snapshots"; out.mkdir(parents=True, exist_ok=True)
suffix = ("" if POSITIVES == "walk" else f"_{POSITIVES}") + ("" if AGG == "mean" else f"_{AGG}")   # each ablation snapshot gets its own file
comp_csv = out / f"{DATASET}_encoder_comparison_K{K}{suffix}.csv"
comp.to_csv(comp_csv, index=False)
print(f"{DATASET} | virtual graph: {SIM}, K={K} | positives: {POSITIVES} | agg: {AGG} | seeds: {SEEDS} | same graph, same splits — only the encoder differs")
print(f"saved -> {comp_csv}")
display(comp)

for enc_name, got in (("deepwalk", bridge), (ENCODER, sage)):   # every run also lands in the accumulating scoreboard
    for t in (NC, LP):
        if got[t]:
            results_io.record_score(DATASET, enc_name, SIM, K, t, SEEDS, got[t])
print("recorded -> results/scoreboard.csv  (section 7 shows every variant tested so far)")

actor | virtual graph: psi, K=10 | positives: edge | agg: mean | seeds: [42, 43, 44] | same graph, same splits — only the encoder differs
saved -> results/snapshots/actor_encoder_comparison_K10_edge.csv


,task,deepwalk_phase2,graphsage_edge_phase3,improvement,winner
0,node classification (weighted F1),0.2023,0.1691,-0.0332,deepwalk
1,link prediction (AUC),0.5034,0.6466,0.1433,graphsage_edge


recorded -> results/scoreboard.csv  (section 7 shows every variant tested so far)


## 7 · Detailed graph/encoder comparison

Tables compare the two headline encoders only: **`deepwalk` vs `graphsage_edge`**.

Workflow: run the sweep (§8) or new Setup knobs (§1–6) → rerun this section.


In [8]:
"""Mode 1 — MAIN RESULTS: deepwalk vs graphsage_edge (the locked ViRGo encoder), one table per task, for EVERY dataset in the batch."""
sb = Path("results") / "scoreboard.csv"
assert sb.exists(), f"{sb} missing -> run section 8 once first (it records the scores)"
board = pd.read_csv(sb)
MAIN = ["deepwalk", "graphsage_edge"]                                            # main comparison: walk baseline vs locked ViRGo encoder (all other encoders -> Mode 2 below)
VG_ORDER = {"psi": 0, "degree": 1, "centrality": 2, "original": 3, "hybrid": 4, "hybrid_degree": 5, "hybrid_centrality": 6}  # row order for the graph variants
SHOW_K = [10]                                       # <-- K knob: which K values to show, e.g. [5] | [10] | [5, 10, 20]
SHOW_DATASETS = RUN_DATASETS                        # <-- which datasets to table; set [DATASET] for the spine graph only
SHOW_CHARTS = False                                 # <-- True also draws one bar chart per dataset x task (2 figures per dataset)

for ds7 in SHOW_DATASETS:
    b = board[(board["dataset"] == ds7) & (board["encoder"].isin(MAIN)) & (board["top_K_neighbors"].isin(SHOW_K))].copy()
    if b.empty:
        print(f"\n{ds7}: no scores at K={SHOW_K} -> run section 8 first")
        continue
    encoders = [e for e in MAIN if e in set(b["encoder"])]
    full = len(encoders) * b.groupby(["graph_variant", "top_K_neighbors"]).ngroups * b["task"].nunique()
    print(f"\n########## {ds7} (K={SHOW_K}): {len(b)} scores"
          + ("  -> complete grid, nothing missing" if len(b) == full else f"  -> INCOMPLETE (expected {full}) - missing combos show as NaN") + " ##########")

    for task in sorted(b["task"].unique(), reverse=True):                        # node classification first, then link prediction
        t = b[b["task"] == task]
        tab = t.pivot_table(index=["graph_variant", "top_K_neighbors"], columns="encoder", values="mean").reindex(columns=encoders)
        tab = tab.reset_index().rename(columns={"top_K_neighbors": "K"}).rename_axis(columns=None)
        tab["_order"] = tab["graph_variant"].map(VG_ORDER).fillna(len(VG_ORDER))
        tab = tab.sort_values(["_order", "K"]).drop(columns="_order").reset_index(drop=True)
        tab["improvement"] = (tab["graphsage_edge"] - tab["deepwalk"]).round(4) if len(encoders) == 2 else np.nan
        tab["winner"] = tab[encoders].idxmax(axis=1)
        print(f"=== {ds7} — {task} ===")
        display(tab.round(4))

    if SHOW_CHARTS:
        b["variant"] = b["graph_variant"] + "\nK=" + b["top_K_neighbors"].astype(str)   # two-line labels keep all groups readable
        for task in sorted(b["task"].unique(), reverse=True):
            t = b[b["task"] == task]
            means = t.pivot_table(index="variant", columns="encoder", values="mean")
            stds = t.pivot_table(index="variant", columns="encoder", values="std")
            ax = means.plot.bar(yerr=stds, figsize=(14, 4.5), capsize=3, title=f"{ds7} — {task} (main results, K={SHOW_K})")
            ax.set_xlabel("virtual graph and K"); ax.set_ylabel(task.split("(")[-1].rstrip(")"))
            ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
            plt.tight_layout(); plt.show()


########## actor (K=[10]): 28 scores  -> complete grid, nothing missing ##########
=== actor — node classification (weighted F1) ===


,graph_variant,K,deepwalk,graphsage_edge,improvement,winner
0,psi,10,0.2023,0.1691,-0.0332,deepwalk
1,degree,10,0.1831,0.1674,-0.0157,deepwalk
2,centrality,10,0.1970,0.1660,-0.0310,deepwalk
3,original,10,0.2075,0.1850,-0.0225,deepwalk
4,hybrid,10,0.2026,0.1728,-0.0298,deepwalk
5,hybrid_degree,10,0.1948,0.1675,-0.0273,deepwalk
6,hybrid_centrality,10,0.1999,0.1677,-0.0322,deepwalk


=== actor — link prediction (AUC) ===


,graph_variant,K,deepwalk,graphsage_edge,improvement,winner
0,psi,10,0.5034,0.6466,0.1432,graphsage_edge
1,degree,10,0.6152,0.6617,0.0465,graphsage_edge
2,centrality,10,0.5041,0.5914,0.0873,graphsage_edge
3,original,10,0.7016,0.5953,-0.1063,deepwalk
4,hybrid,10,0.6329,0.6409,0.0080,graphsage_edge
5,hybrid_degree,10,0.5996,0.6514,0.0518,graphsage_edge
6,hybrid_centrality,10,0.6368,0.6228,-0.0140,deepwalk



########## amazon_photo (K=[10]): 28 scores  -> complete grid, nothing missing ##########
=== amazon_photo — node classification (weighted F1) ===


...

,graph_variant,K,deepwalk,graphsage_edge,improvement,winner
0,psi,10,0.5833,0.7609,0.1776,graphsage_edge
1,degree,10,0.5826,0.7736,0.1910,graphsage_edge
2,centrality,10,0.6070,0.7728,0.1658,graphsage_edge
3,original,10,0.8985,0.7399,-0.1586,deepwalk
4,hybrid,10,0.9332,0.7542,-0.1790,deepwalk
5,hybrid_degree,10,0.9094,0.7563,-0.1531,deepwalk
6,hybrid_centrality,10,0.9400,0.7480,-0.1920,deepwalk



########## tolokers (K=[10]): 26 scores  -> INCOMPLETE (expected 28) - missing combos show as NaN ##########
=== tolokers — node classification (weighted F1) ===


,graph_variant,K,deepwalk,graphsage_edge,improvement,winner
0,psi,10,0.6897,0.7129,0.0232,graphsage_edge
1,degree,10,0.6907,0.7158,0.0251,graphsage_edge
2,centrality,10,0.6877,0.7132,0.0255,graphsage_edge
3,original,10,0.7543,0.7345,-0.0198,deepwalk
4,hybrid,10,0.7443,0.7265,-0.0178,deepwalk
5,hybrid_degree,10,0.7401,0.7249,-0.0152,deepwalk


=== tolokers — link prediction (AUC) ===


,graph_variant,K,deepwalk,graphsage_edge,improvement,winner
0,psi,10,0.4751,0.7007,0.2256,graphsage_edge
1,degree,10,0.4469,0.6936,0.2467,graphsage_edge
2,centrality,10,0.5142,0.6880,0.1738,graphsage_edge
3,original,10,0.7859,0.6444,-0.1415,deepwalk
4,hybrid,10,0.8954,0.6831,-0.2123,deepwalk
5,hybrid_degree,10,0.8959,0.6833,-0.2126,deepwalk
6,hybrid_centrality,10,0.8837,0.6704,-0.2133,deepwalk


**Reading the tables:** encoder columns = mean over `SEEDS` · `graph_variant`: `psi` (I2V Poisson/KL) | `degree` | `centrality` | `original` (exact copy of the input graph — control) | `hybrid` (original ∪ psi top-K) | `hybrid_degree` / `hybrid_centrality` (original ∪ degree/centrality top-K) · `K` = top-K neighbors · `improvement` = `graphsage_edge` − `deepwalk` · `winner` = row's best encoder.


## 8 · Conclusion — which virtual graph is best?

Reuse-or-create ⇒ interruptible: rerunning resumes where it stopped. When done, rerun **§7**.


In [9]:
""" locked encoder over every (dataset, graph variant, K) — trains what's missing, records each config to the scoreboard."""
SWEEP_DATASETS = RUN_DATASETS      # <-- the batch comes from Setup, same list as notebook 2; local ds8/G8 vars keep the spine's DATASET/G untouched
SWEEP_SIMS = ["psi", "degree", "centrality", "original", "hybrid", "hybrid_degree", "hybrid_centrality"]        # <-- graph variants to sweep (last two = local extra hybrids, NOT in cfg.VG_SIMS)
SWEEP_KS = [10]                              # <-- K values to sweep (full grid; set [10] for a quick single-K pass)

NC, LP = "node classification (weighted F1)", "link prediction (AUC)"

for ds8 in SWEEP_DATASETS:
    G8 = graph_io.load_graph(cfg.dataset(ds8)["edgelist"])
    labels8 = cfg.dataset(ds8)["labels"]
    have8 = labels8 is not None and Path(labels8).exists()
    print(f"===== {ds8}: {G8.number_of_nodes()} nodes / {G8.number_of_edges()} edges =====", flush=True)
    for sim in SWEEP_SIMS:
        for k in SWEEP_KS:
            vpath = NB2 / "virtual_graphs" / ds8 / f"k{k}" / sim / "virtual_graph.edgelist"
            if not vpath.exists():                     # skip, never crash: one unprepared dataset must not kill the whole batch
                print(f"[{ds8} {sim} K={k}] SKIPPED - {vpath} missing -> run notebook 2 for this dataset/variant first", flush=True)
                continue
            Vs = nx.read_weighted_edgelist(vpath, nodetype=int)
            Vs.add_nodes_from(G8.nodes)                 # restore isolated nodes dropped by the edgelist format
            got = {NC: [], LP: []}
            for seed in SEEDS:
                split_dir = LP_SPLITS / ds8 / f"seed_{seed}"
                emb = NB3 / "node_classification" / ds8 / f"k{k}" / sim / f"{ENCODER}_s{seed}.emb"
                if not emb.exists() or FORCE_REBUILD:
                    print(f"[{ds8} {sim} K={k}] train NC seed {seed} ...", flush=True)
                    emb.parent.mkdir(parents=True, exist_ok=True)
                    enc = SageEncoder(G8, Vs, seed, hidden=P["hidden"], dimensions=P["dimensions"], layers=P["layers"], agg=AGG, feats=FEATURES)
                    enc.train(P["epochs"], lr=P["lr"], negatives=P["negatives"],
                              pairs_per_epoch=P["pairs_per_epoch"], max_pairs=P["max_pairs"], positives=POSITIVES)
                    enc.save(emb)
                if have8:
                    f1s, _, _ = eval_nodeclass.evaluate(str(emb), str(labels8), train_frac=cfg.REPRO["nodeclass_train_frac"], seed=seed)
                    got[NC].append(f1s["weighted"])
                lemb = NB3 / "link_prediction" / ds8 / f"k{k}" / sim / f"{ENCODER}_s{seed}.emb"
                if not lemb.exists() or FORCE_REBUILD:
                    print(f"[{ds8} {sim} K={k}] train LP seed {seed} ...", flush=True)
                    lemb.parent.mkdir(parents=True, exist_ok=True)
                    Gt = graph_io.load_graph(split_dir / "train.edgelist")
                    enc = SageEncoder(Gt, VirtualGraph(Gt, seed=cfg.REPRO["seed"]).build(sim, k), seed, hidden=P["hidden"], dimensions=P["dimensions"],
                                      layers=P["layers"], agg=AGG, feats=FEATURES)
                    enc.train(P["epochs"], lr=P["lr"], negatives=P["negatives"],
                              pairs_per_epoch=P["pairs_per_epoch"], max_pairs=P["max_pairs"], positives=POSITIVES)
                    enc.save(lemb)
                got[LP].append(eval_linkpred.evaluate(str(lemb), str(split_dir), seed=seed, score=cfg.REPRO["linkpred_score"]))
            for t in (NC, LP):
                if got[t]:
                    results_io.record_score(ds8, ENCODER, sim, k, t, SEEDS, got[t])
            print(f"[{ds8} {sim} K={k}] " + (f"NC {np.mean(got[NC]):.4f}  " if got[NC] else "") + f"LP {np.mean(got[LP]):.4f}  -> scoreboard")

            vg = {NC: [], LP: []}                       # deepwalk baseline row: score the SAVED Phase-2 bridge embeddings, never train here
            for seed in SEEDS:
                nemb = NB2 / "node_classification" / ds8 / f"k{k}" / sim / f"deepwalk_s{seed}.emb"
                lemb = NB2 / "link_prediction" / ds8 / f"k{k}" / sim / f"deepwalk_s{seed}.emb"
                if have8 and nemb.exists():
                    f1s, _, _ = eval_nodeclass.evaluate(str(nemb), str(labels8), train_frac=cfg.REPRO["nodeclass_train_frac"], seed=seed)
                    vg[NC].append(f1s["weighted"])
                if lemb.exists():
                    vg[LP].append(eval_linkpred.evaluate(str(lemb), str(LP_SPLITS / ds8 / f"seed_{seed}"),
                                                         seed=seed, score=cfg.REPRO["linkpred_score"]))
            for t in (NC, LP):
                if len(vg[t]) == len(SEEDS):            # record only complete seed sets; partial/missing -> run notebook 2 for this sim/K
                    results_io.record_score(ds8, "deepwalk", sim, k, t, SEEDS, vg[t])
            if len(vg[LP]) == len(SEEDS):
                print(f"[{ds8} {sim} K={k}] deepwalk " + (f"NC {np.mean(vg[NC]):.4f}  " if len(vg[NC]) == len(SEEDS) else "") + f"LP {np.mean(vg[LP]):.4f}  -> scoreboard")
            else:
                print(f"[{ds8} {sim} K={k}] deepwalk skipped (bridge embeddings missing -> notebook 2)")
print("sweep done — rerun section 7 to see all variants side by side")

===== actor: 7600 nodes / 26659 edges =====
[actor psi K=10] NC 0.1691  LP 0.6466  -> scoreboard
[actor psi K=10] deepwalk NC 0.2023  LP 0.5034  -> scoreboard
[actor degree K=10] NC 0.1674  LP 0.6617  -> scoreboard
[actor degree K=10] deepwalk NC 0.1831  LP 0.6152  -> scoreboard
[actor centrality K=10] NC 0.1660  LP 0.5914  -> scoreboard
[actor centrality K=10] deepwalk NC 0.1970  LP 0.5041  -> scoreboard
[actor original K=10] NC 0.1850  LP 0.5953  -> scoreboard
[actor original K=10] deepwalk NC 0.2075  LP 0.7016  -> scoreboard
[actor hybrid K=10] NC 0.1728  LP 0.6409  -> scoreboard
[actor hybrid K=10] deepwalk NC 0.2026  LP 0.6329  -> scoreboard
[actor hybrid_degree K=10] NC 0.1675  LP 0.6514  -> scoreboard
[actor hybrid_degree K=10] deepwalk NC 0.1948  LP 0.5996  -> scoreboard
[actor hybrid_centrality K=10] NC 0.1677  LP 0.6228  -> scoreboard
[actor hybrid_centrality K=10] deepwalk NC 0.1999  LP 0.6368  -> scoreboard
===== amazon_photo: 7650 nodes / 119081 edges =====
[amazon_photo p

## 8b · Ablation D — features or message passing?

GraphSAGE beats the bridge with **two** advantages at once: message passing _and_ four structural input features DeepWalk's lookup table cannot consume. This section separates them.

| id     | features                             | message passing | dims | question                                                                |
| ------ | ------------------------------------ | --------------- | ---- | ----------------------------------------------------------------------- |
| **D0** | degree + centrality + ψ + clustering | ✔ 2-layer       | 4    | the locked ViRGo encoder (**reused from disk, not retrained**)          |
| **D1** | degree                               | ✔               | 1    | is degree alone enough?                                                 |
| **D2** | degree + centrality                  | ✔               | 2    | does centrality add anything?                                           |
| **D3** | ψ                                    | ✔               | 1    | is the I2V scalar enough? _(confounded — the ψ graph was built from ψ)_ |
| **D4** | seeded random                        | ✔               | 4    | **control: message passing with zero structural signal**                |
| **D5** | constant                             | ✔               | 1    | floor — identical rows collapse to zeros, expect AUC ≈ 0.50             |
| **D6** | degree + centrality + ψ + clustering | ✘ `layers=0`    | 4    | **control: the features with zero message passing**                     |

**The 2×2 this closes.** D0 = features + passing · D4 = passing, no features · D6 = features, no passing.

- **D4** answers _are the features necessary?_ — at/below the bridge ⇒ yes, passing alone is not sufficient.
- **D6** answers _does passing add anything?_ — **D6 ≈ D0** ⇒ the GNN adds nothing, the win was the features · **D6 « D0** ⇒ passing earns its place on top of the features.

Neither control alone decides it; the pair does. D6 trains nothing — `layers=0` means the z-normed features **are** the embedding (same feature builder as D0, computed on the 70% train graph for LP ⇒ no leakage).

Artifacts: `graphsage_edge_feat_<set>_s<seed>.emb` and `features_only_s<seed>.emb`; scoreboard encoder names likewise. Snapshot → `results/snapshots/<dataset>_feature_ablation_K<K>.csv`.


In [10]:
"""Ablation D: same graph, same encoder, only the input features change; D6 removes message passing entirely. Trains what's missing, records to the scoreboard."""
D_SETS = ["all", "degree", "deg_cent", "psi", "centrality", "clustering", "random", "const"]   # <-- feature sets to run ("all" = D0, reused from disk; "random" = the control)
D_DIMS = {"all": 4, "degree": 1, "deg_cent": 2, "psi": 1, "centrality": 1, "clustering": 1, "random": 4, "const": 1}
RUN_D6 = True                                       # <-- D6: raw features straight to the classifier, no GNN — the arm that decides features-vs-message-passing

labels = cfg.dataset(DATASET)["labels"]
have_labels = labels is not None and Path(labels).exists()
NC, LP = "node classification (weighted F1)", "link prediction (AUC)"
vpath = NB2 / "virtual_graphs" / DATASET / f"k{K}" / SIM / "virtual_graph.edgelist"
assert vpath.exists(), f"{vpath} missing -> run notebooks/2-phase_2_virtual_graph.ipynb for sim={SIM}, K={K} first"
Vd = nx.read_weighted_edgelist(vpath, nodetype=int)
Vd.add_nodes_from(G.nodes)                          # restore isolated nodes dropped by the edgelist format

d_rows = []
for feats in D_SETS:
    enc_name = "graphsage_edge" + ("" if feats == "all" else f"_feat_{feats}")   # D0 keeps the locked name -> its embeddings are reused, not retrained
    label, desc = cfg.D_FEATURES[feats]
    got = {NC: [], LP: []}
    for seed in SEEDS:
        split_dir = LP_SPLITS / DATASET / f"seed_{seed}"
        emb = NB3 / "node_classification" / DATASET / f"k{K}" / SIM / f"{enc_name}_s{seed}.emb"
        if not emb.exists() or FORCE_REBUILD:
            print(f"[{label}] train NC seed {seed} ...")
            enc = SageEncoder(G, Vd, seed, hidden=P["hidden"], dimensions=P["dimensions"], layers=P["layers"], agg=AGG, feats=feats)
            enc.train(P["epochs"], lr=P["lr"], negatives=P["negatives"],
                      pairs_per_epoch=P["pairs_per_epoch"], max_pairs=P["max_pairs"], positives=POSITIVES)
            enc.save(emb)
        if have_labels:
            f1s, _, _ = eval_nodeclass.evaluate(str(emb), str(labels), train_frac=cfg.REPRO["nodeclass_train_frac"], seed=seed)
            got[NC].append(f1s["weighted"])
        lemb = NB3 / "link_prediction" / DATASET / f"k{K}" / SIM / f"{enc_name}_s{seed}.emb"
        if not lemb.exists() or FORCE_REBUILD:
            print(f"[{label}] train LP seed {seed} ...")
            Gt = graph_io.load_graph(split_dir / "train.edgelist")
            enc = SageEncoder(Gt, VirtualGraph(Gt, seed=cfg.REPRO["seed"]).build(SIM, K), seed, hidden=P["hidden"], dimensions=P["dimensions"],
                              layers=P["layers"], agg=AGG, feats=feats)   # features come from the 70% train graph -> no leakage
            enc.train(P["epochs"], lr=P["lr"], negatives=P["negatives"],
                      pairs_per_epoch=P["pairs_per_epoch"], max_pairs=P["max_pairs"], positives=POSITIVES)
            enc.save(lemb)
        got[LP].append(eval_linkpred.evaluate(str(lemb), str(split_dir), seed=seed, score=cfg.REPRO["linkpred_score"]))
    for t in (NC, LP):
        if got[t]:
            results_io.record_score(DATASET, enc_name, SIM, K, t, SEEDS, got[t])
    d_rows.append({"feature_set": label, "dims": D_DIMS[feats],
                   "NC F1": np.mean(got[NC]) if got[NC] else np.nan, "NC std": np.std(got[NC]) if got[NC] else np.nan,
                   "LP AUC": np.mean(got[LP]), "LP std": np.std(got[LP]), "conclusion": desc})
    print(f"[{label}] " + (f"NC {np.mean(got[NC]):.4f}  " if got[NC] else "") + f"LP {np.mean(got[LP]):.4f}  -> scoreboard")

if RUN_D6:                                          # D6: layers=0 -> no convs, the z-normed features ARE the embedding; nothing to train, virtual graph unused
    label, desc = cfg.D_FEATURES["none_mp"]
    got = {NC: [], LP: []}
    for seed in SEEDS:
        split_dir = LP_SPLITS / DATASET / f"seed_{seed}"
        emb = NB3 / "node_classification" / DATASET / f"k{K}" / SIM / f"features_only_s{seed}.emb"
        if not emb.exists() or FORCE_REBUILD:
            print(f"[{label}] build NC seed {seed} (no training) ...")
            SageEncoder(G, G, seed, layers=0, feats="all").save(emb)          # same feature builder as D0, message passing removed
        if have_labels:
            f1s, _, _ = eval_nodeclass.evaluate(str(emb), str(labels), train_frac=cfg.REPRO["nodeclass_train_frac"], seed=seed)
            got[NC].append(f1s["weighted"])
        lemb = NB3 / "link_prediction" / DATASET / f"k{K}" / SIM / f"features_only_s{seed}.emb"
        if not lemb.exists() or FORCE_REBUILD:
            print(f"[{label}] build LP seed {seed} (no training) ...")
            Gt = graph_io.load_graph(split_dir / "train.edgelist")
            SageEncoder(Gt, Gt, seed, layers=0, feats="all").save(lemb)       # features from the 70% train graph -> no leakage
        got[LP].append(eval_linkpred.evaluate(str(lemb), str(split_dir), seed=seed, score=cfg.REPRO["linkpred_score"]))
    for t in (NC, LP):
        if got[t]:
            results_io.record_score(DATASET, "features_only", SIM, K, t, SEEDS, got[t])
    d_rows.append({"feature_set": label, "dims": 4,
                   "NC F1": np.mean(got[NC]) if got[NC] else np.nan, "NC std": np.std(got[NC]) if got[NC] else np.nan,
                   "LP AUC": np.mean(got[LP]), "LP std": np.std(got[LP]), "conclusion": desc})
    print(f"[{label}] " + (f"NC {np.mean(got[NC]):.4f}  " if got[NC] else "") + f"LP {np.mean(got[LP]):.4f}  -> scoreboard")

sb = pd.read_csv(Path("results") / "scoreboard.csv")                             # reference row: read fresh, never retrained here
ref = sb[(sb["dataset"] == DATASET) & (sb["encoder"] == "deepwalk") & (sb["graph_variant"] == SIM) & (sb["top_K_neighbors"] == K)]
ref = {r["task"]: r for _, r in ref.iterrows()}
d = pd.DataFrame([{"feature_set": "deepwalk", "dims": np.nan,
                   "NC F1": ref[NC]["mean"] if NC in ref else np.nan, "NC std": ref[NC]["std"] if NC in ref else np.nan,
                   "LP AUC": ref[LP]["mean"] if LP in ref else np.nan, "LP std": ref[LP]["std"] if LP in ref else np.nan,
                   "conclusion": "reference: no features, no message passing"}] + d_rows)
d0 = d[d["feature_set"] == cfg.D_FEATURES["all"][0]]                             # deltas are measured against the locked encoder (D0)
d["Δ NC"] = (d["NC F1"] - float(d0["NC F1"].iloc[0])).round(4) if len(d0) else np.nan
d["Δ LP"] = (d["LP AUC"] - float(d0["LP AUC"].iloc[0])).round(4) if len(d0) else np.nan

out = Path("results") / "snapshots"; out.mkdir(parents=True, exist_ok=True)
d_csv = out / f"{DATASET}_feature_ablation_K{K}.csv"
d.to_csv(d_csv, index=False)
print(f"\n{DATASET} | ablation D | virtual graph: {SIM}, K={K} | positives: {POSITIVES} | agg: {AGG} | seeds: {SEEDS} — only the input features differ (D6: no message passing)")
print(f"saved -> {d_csv}")
display(d[["feature_set", "dims", "NC F1", "NC std", "LP AUC", "LP std", "Δ NC", "Δ LP", "conclusion"]].round(4))
print("read D4 random vs deepwalk: above the bridge => message passing carries signal; at/below => features are doing the work")
print("read D6 vs D0 (Δ LP): ~0 => message passing adds nothing, the win is the features; strongly negative => the GNN earns its place")

[D0 all] NC 0.1691  LP 0.6466  -> scoreboard
[D1 degree] NC 0.1252  LP 0.5223  -> scoreboard
[D2 deg+cent] NC 0.1590  LP 0.5891  -> scoreboard
[D3 psi] NC 0.1076  LP 0.4633  -> scoreboard
[D7 centrality] NC 0.1466  LP 0.5470  -> scoreboard
[D8 clustering] NC 0.1498  LP 0.5882  -> scoreboard
[D4 random] NC 0.1889  LP 0.4963  -> scoreboard
[D5 constant] NC 0.1061  LP 0.4923  -> scoreboard
[D6 features only] NC 0.1602  LP 0.5871  -> scoreboard

actor | ablation D | virtual graph: psi, K=10 | positives: edge | agg: mean | seeds: [42, 43, 44] — only the input features differ (D6: no message passing)
saved -> results/snapshots/actor_feature_ablation_K10.csv


,feature_set,dims,NC F1,NC std,LP AUC,LP std,Δ NC,Δ LP,conclusion
0,deepwalk,NaN,0.2023,0.0030,0.5034,0.0061,0.0332,-0.1432,"reference: no features, no message passing"
1,D0 all,4.0,0.1691,0.0053,0.6466,0.0035,0.0000,0.0000,degree + centrality + psi + clustering
2,D1 degree,1.0,0.1252,0.0082,0.5223,0.0182,-0.0439,-0.1243,degree only
3,D2 deg+cent,2.0,0.1590,0.0077,0.5891,0.0089,-0.0101,-0.0575,degree + eigenvector centrality
4,D3 psi,1.0,0.1076,0.0008,0.4633,0.0036,-0.0614,-0.1833,psi only (confounded: the psi graph was built ...
5,D7 centrality,1.0,0.1466,0.0088,0.5470,0.0072,-0.0225,-0.0996,eigenvector centrality only (per-component nor...
6,D8 clustering,1.0,0.1498,0.0191,0.5882,0.0108,-0.0193,-0.0585,local clustering coefficient only
7,D4 random,4.0,0.1889,0.0090,0.4963,0.0058,0.0198,-0.1503,seeded random features - control: message pass...
8,D5 constant,1.0,0.1061,0.0000,0.4923,0.0285,-0.0630,-0.1544,"identical rows -> z-norm zeros - floor, expect..."
9,D6 features only,4.0,0.1602,0.0097,0.5871,0.0238,-0.0089,-0.0595,"raw features, no message passing (layers=0) - ..."


read D4 random vs deepwalk: above the bridge => message passing carries signal; at/below => features are doing the work
read D6 vs D0 (Δ LP): ~0 => message passing adds nothing, the win is the features; strongly negative => the GNN earns its place


## 09 · Cross-dataset view

All datasets in the scoreboard **side by side** (columns = dataset × encoder, rows = graph variant). Reads `results/scoreboard.csv` directly — independent of the `DATASET` knob. Display only.


In [11]:
"""Cross-dataset view: every dataset in the scoreboard side by side, main encoders only."""
XD_K = [10]                                         # <-- K knob: K=10 is the finalized setting (every dataset has the full grid there); [5, 10, 20] shows the cora-only sweep too
MAIN = ["deepwalk", "graphsage_edge"]
VG_ORDER = {"psi": 0, "degree": 1, "centrality": 2, "original": 3, "hybrid": 4, "hybrid_degree": 5, "hybrid_centrality": 6}

xb = pd.read_csv(Path("results") / "scoreboard.csv")
x = xb[(xb["encoder"].isin(MAIN)) & (xb["top_K_neighbors"].isin(XD_K))].copy()
assert not x.empty, f"no scores at K={XD_K} -> pick another XD_K, or run section 8 for that K first"
datasets = sorted(x["dataset"].unique())
print(f"cross-dataset (K={XD_K}): datasets={datasets}, {len(x)} scores — missing combos show as NaN")

for task in sorted(x["task"].unique(), reverse=True):                            # node classification first, then link prediction
    t = x[x["task"] == task]
    tab = t.pivot_table(index=["graph_variant", "top_K_neighbors"], columns=["dataset", "encoder"], values="mean")
    tab = tab.reindex(columns=pd.MultiIndex.from_product([datasets, MAIN]))      # fixed column order, gaps stay visible
    tab = tab.reindex([s for s in VG_ORDER if s in tab.index.get_level_values(0)], level=0)
    tab = tab.dropna(axis=1, how="all")     # drop dataset x encoder columns with NO score for THIS task: ogbl_ddi/ogbn_arxiv
                                            # are scored under "(OGB official)", so they would show as all-NaN columns here
    tab.index.names = ["graph_variant", "K"]
    print(f"\n=== {task} ===")
    display(tab.round(4))

cross-dataset (K=[10]): datasets=['actor', 'amazon_photo', 'amazon_ratings', 'citeseer_linqs', 'cora', 'enzymes', 'lastfm_asia', 'minesweeper', 'ogbl_ddi', 'ogbn_arxiv', 'proteins', 'pubmed', 'questions', 'roman_empire', 'squirrel_filtered', 'tolokers'], 455 scores — missing combos show as NaN

=== node classification (weighted F1) ===


actor                amazon_photo                 \
                     deepwalk graphsage_edge     deepwalk graphsage_edge   
graph_variant     K                                                        
psi               10   0.2023         0.1691       0.2292         0.4195   
degree            10   0.1831         0.1674       0.1887         0.3759   
centrality        10   0.1970         0.1660       0.3477         0.4667   
original          10   0.2075         0.1850       0.9180         0.6174   
hybrid            10   0.2026         0.1728       0.8253         0.5301   
hybrid_degree     10   0.1948         0.1675       0.8437         0.5177   
hybrid_centrality 10   0.1999         0.1677       0.8238         0.5689   

                     amazon_ratings                citeseer_linqs  \
                           deepwalk graphsage_edge       deepwalk   
graph_variant     K                                                 
psi               10         0.2308         0.2546         0.2255   
degree            10         0.2123         0.2543         0.1931   
centrality        10         0.2488         0.2469         0.3533   
original          10         0.3447         0.2875         0.5473   
hybrid            10         0.2616         0.2623         0.3146   
hybrid_degree     10         0.2860         0.2662         0.2860   
hybrid_centrality 10         0.2881         0.2689         0.4099   

                                        cora                 ...   pubmed  \
                     graphsage_edge deepwalk graphsage_edge  ... deepwalk   
graph_variant     K                                          ...            
psi               10         0.2327   0.2133         0.2351  ...   0.4191   
degree            10         0.2291   0.1590         0.2351  ...   0.3527   
centrality        10         0.2818   0.3666         0.2942  ...   0.4118   
original          10         0.3298   0.8100         0.4266  ...   0.8036   
hybrid            10         0.2698   0.4469         0.2918  ...   0.5219   
hybrid_degree     10         0.2597   0.4597         0.2661  ...   0.5214   
hybrid_centrality 10         0.3069   0.5225         0.3554  ...   0.5955   

                                    questions                roman_empire  \
                     graphsage_edge  deepwalk graphsage_edge     deepwalk   
graph_variant     K                                                         
psi               10         0.4717    0.9555         0.9554       0.0906   
degree            10         0.4674    0.9555         0.9554       0.1353   
centrality        10         0.5151    0.9555         0.9554       0.0848   
original          10         0.5958    0.9555         0.9555       0.1207   
hybrid            10         0.5123    0.9555         0.9555       0.1064   
hybrid_degree     10         0.4999    0.9555         0.9555       0.1370   
hybrid_centrality 10         0.5629    0.9555         0.9555       0.0897   

                                    squirrel_filtered                tolokers  \
                     graphsage_edge          deepwalk graphsage_edge deepwalk   
graph_variant     K                                                             
psi               10         0.2069            0.2994         0.3500   0.6897   
degree            10         0.2108            0.3156         0.3413   0.6907   
centrality        10         0.2093            0.2871         0.3119   0.6877   
original          10         0.2561            0.2910         0.3358   0.7543   
hybrid            10         0.2144            0.2937         0.3336   0.7443   
hybrid_degree     10         0.2169            0.2838         0.3313   0.7401   
hybrid_centrality 10         0.2158            0.2799         0.3264   0.7391   

                                     
                     graphsage_edge  
graph_variant     K                  
psi               10         0.7129  
degree            10         0.7158  
centrality        10         


=== node classification (OGB official) ===


ogbn_arxiv               
                   deepwalk graphsage_edge
graph_variant K                           
psi           10     0.0492         0.1710
degree        10     0.0784         0.1667
centrality    10     0.0890         0.1822
original      10     0.5985         0.2874
hybrid        10     0.3386         0.2627


=== link prediction (OGB official) ===


ogbl_ddi               
                 deepwalk graphsage_edge
graph_variant K                         
psi           10   0.0039         0.0168
degree        10   0.0039         0.0140
centrality    10   0.0047         0.0211
original      10   0.0322         0.0146
hybrid        10   0.0393         0.0228


=== link prediction (AUC) ===


actor                amazon_photo                 \
                     deepwalk graphsage_edge     deepwalk graphsage_edge   
graph_variant     K                                                        
psi               10   0.5034         0.6466       0.5058         0.7112   
degree            10   0.6152         0.6617       0.4801         0.6661   
centrality        10   0.5041         0.5914       0.5197         0.6918   
original          10   0.7016         0.5953       0.9585         0.7857   
hybrid            10   0.6329         0.6409       0.9542         0.7803   
hybrid_degree     10   0.5996         0.6514       0.9328         0.7736   
hybrid_centrality 10   0.6368         0.6228       0.9505         0.7809   

                     amazon_ratings                citeseer_linqs  \
                           deepwalk graphsage_edge       deepwalk   
graph_variant     K                                                 
psi               10         0.5073         0.5654         0.5331   
degree            10         0.5491         0.4551         0.5410   
centrality        10         0.7112         0.6259         0.6593   
original          10         0.9982         0.7536         0.9234   
hybrid            10         0.9407         0.6550         0.6504   
hybrid_degree     10         0.7971         0.5102         0.5975   
hybrid_centrality 10         0.9220         0.6949         0.8232   

                                        cora                 ...   pubmed  \
                     graphsage_edge deepwalk graphsage_edge  ... deepwalk   
graph_variant     K                                          ...            
psi               10         0.5057   0.4990         0.5150  ...   0.5055   
degree            10         0.5415   0.5501         0.5391  ...   0.6003   
centrality        10         0.5437   0.5634         0.5659  ...   0.5044   
original          10         0.6218   0.8971         0.6130  ...   0.9214   
hybrid            10         0.5166   0.6738         0.5463  ...   0.6840   
hybrid_degree     10         0.5598   0.5587         0.5505  ...   0.6358   
hybrid_centrality 10         0.5658   0.7887         0.5904  ...   0.7299   

                                    questions                roman_empire  \
                     graphsage_edge  deepwalk graphsage_edge     deepwalk   
graph_variant     K                                                         
psi               10         0.5897    0.5750         0.4923       0.5703   
degree            10         0.5449    0.6112         0.5272       0.4774   
centrality        10         0.6221    0.5008         0.5078       0.6875   
original          10         0.6392    0.6589         0.4665       0.9994   
hybrid            10         0.6238    0.6537         0.5526       0.8065   
hybrid_degree     10         0.5637    0.6315         0.5703       0.5550   
hybrid_centrality 10         0.6624    0.6816         0.5515       0.8095   

                                    squirrel_filtered                tolokers  \
                     graphsage_edge          deepwalk graphsage_edge deepwalk   
graph_variant     K                                                             
psi               10         0.5407            0.5833         0.7609   0.4751   
degree            10         0.5092            0.5826         0.7736   0.4469   
centrality        10         0.6980            0.6070         0.7728   0.5142   
original          10         0.6019            0.8985         0.7399   0.7859   
hybrid            10         0.5408            0.9332         0.7542   0.8954   
hybrid_degree     10         0.5093            0.9094         0.7563   0.8959   
hybrid_centrality 10         0.6931            0.9400         0.7480   0.8837   

                                     
                     graphsage_edge  
graph_variant     K                  
psi               10         0.7007  
degree            10         0.6936  
centrality        10         

## 10 · Research question — which virtual graph is best?

The **virtual graph is the variable under study** — so the encoder is held fixed at the locked **`graphsage_edge`** (changing graph _and_ encoder together would confound which one caused the result). `original` = **control baseline**, not a ViRGo contribution.

- **Table 1 (PRIMARY):** best virtual/derived graph (psi / degree / centrality / hybrid) per dataset × task, GraphSAGE only — `best_K` names the winning K.
- **Table 2 (CONTROL):** original vs the best virtual graph, GraphSAGE on both sides — does rewiring beat the real edges at all?
- **Secondary question (encoder):** does GraphSAGE beat DeepWalk on the _same_ graph?


In [12]:
"""ViRGo question under the LOCKED encoder: Table 1 = best VIRTUAL graph per dataset x task (graphsage_edge only); Table 2 = control check vs original."""
RQ_K = [10]                                         # <-- K knob: K=10 is the finalized setting (every dataset has the full grid there); [5, 10, 20] picks the best over the cora-only sweep
RQ_ENCODER = "graphsage_edge"                       # encoder held fixed (locked via ablations A+B); the encoder question lives in section 7, not here

rq = pd.read_csv(Path("results") / "scoreboard.csv")
rq = rq[(rq["encoder"] == RQ_ENCODER) & (rq["top_K_neighbors"].isin(RQ_K))].copy()
assert not rq.empty, f"no {RQ_ENCODER} scores at K={RQ_K} -> pick another RQ_K, or run section 8 for that K first"
rq["task"] = rq["task"].map({"node classification (weighted F1)": "NC (F1)", "link prediction (AUC)": "LP (AUC)"})

# Table 1 — PRIMARY: best virtual/derived graph (psi, degree, centrality, hybrid) per dataset x task, encoder fixed; best over the K sweep.
virt = rq[rq["graph_variant"] != "original"]
best_v = virt.loc[virt.groupby(["dataset", "task"])["mean"].idxmax(),
                  ["dataset", "task", "graph_variant", "top_K_neighbors", "mean"]].reset_index(drop=True)
t1 = best_v.rename(columns={"graph_variant": "best_virtual_graph", "top_K_neighbors": "best_K", "mean": "graphsage_score"})
print(f"Table 1 — PRIMARY: which VIRTUAL graph is best per dataset x task? (encoder fixed = {RQ_ENCODER}; best over K={RQ_K}; original excluded = control)")
display(t1.round(4))

# Table 2 — CONTROL: original graph vs the best virtual graph, same locked encoder on both sides (original ignores K -> max over its rows).
orig = rq[rq["graph_variant"] == "original"].groupby(["dataset", "task"])["mean"].max()
t2 = best_v.set_index(["dataset", "task"])["mean"].rename("best_virtual_score").to_frame()
t2["original_score"] = orig
t2 = t2.reset_index()[["dataset", "task", "original_score", "best_virtual_score"]]
print(f"\nTable 2 — CONTROL check ({RQ_ENCODER} on both sides): does the original graph beat the best virtual graph?")
display(t2.round(4))

Table 1 — PRIMARY: which VIRTUAL graph is best per dataset x task? (encoder fixed = graphsage_edge; best over K=[10]; original excluded = control)


,dataset,task,best_virtual_graph,best_K,graphsage_score
0,actor,LP (AUC),degree,10,0.6617
1,actor,NC (F1),hybrid,10,0.1728
2,amazon_photo,LP (AUC),hybrid_centrality,10,0.7809
3,amazon_photo,NC (F1),hybrid_centrality,10,0.5689
4,amazon_ratings,LP (AUC),hybrid_centrality,10,0.6949
5,amazon_ratings,NC (F1),hybrid_centrality,10,0.2689
6,citeseer_linqs,LP (AUC),hybrid_centrality,10,0.5658
7,citeseer_linqs,NC (F1),hybrid_centrality,10,0.3069
8,cora,LP (AUC),hybrid_centrality,10,0.5904
9,cora,NC (F1),hybrid_centrality,10,0.3554



Table 2 — CONTROL check (graphsage_edge on both sides): does the original graph beat the best virtual graph?


,dataset,task,original_score,best_virtual_score
0,actor,LP (AUC),0.5953,0.6617
1,actor,NC (F1),0.1850,0.1728
2,amazon_photo,LP (AUC),0.7857,0.7809
3,amazon_photo,NC (F1),0.6174,0.5689
4,amazon_ratings,LP (AUC),0.7536,0.6949
5,amazon_ratings,NC (F1),0.2875,0.2689
6,citeseer_linqs,LP (AUC),0.6218,0.5658
7,citeseer_linqs,NC (F1),0.3298,0.3069
8,cora,LP (AUC),0.6130,0.5904
9,cora,NC (F1),0.4266,0.3554
